# Support Vector Machines (SVMs) Tutorial
### Dr. Ye Kyaw Thu, Lab Leader, Language Understanding Lab., AI Mario, Myanmar
### for AI Engineering (Fundamental) Class Students
### Date: 28 July 2026
### Last Updated: 16 July 2026

Support Vector Machines with Applications: [https://arxiv.org/pdf/math/0612817](https://arxiv.org/pdf/math/0612817)  
Basic Tenets of Classification Algorithms K-Nearest-Neighbor, Support Vector Machine, Random Forest and Neural Network: A Review: [https://www.scirp.org/journal/paperinformation?paperid=104256](https://www.scirp.org/journal/paperinformation?paperid=104256)  
A Training Algorithm for Optimal Margin Classifiers, Presentation Slide: [https://www.eecs.yorku.ca/course_archive/2017-18/F/6412/reading/slides/SVM-wenxiaofu.pdf](https://www.eecs.yorku.ca/course_archive/2017-18/F/6412/reading/slides/SVM-wenxiaofu.pdf)  
Liblinear GitHub Link: [https://github.com/cjlin1/liblinear](https://github.com/cjlin1/liblinear)  
LIBSVM page: [https://www.csie.ntu.edu.tw/~cjlin/libsvm/](https://www.csie.ntu.edu.tw/~cjlin/libsvm/)  
A Practical Guide to Support Vector Classification: [https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf](https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf)  
Wiki SVM basics: [http://en.wikipedia.org/wiki/Support_vector_machine](http://en.wikipedia.org/wiki/Support_vector_machine)  

Dataset myPOS Information: [https://github.com/ye-kyaw-thu/myPOS](https://github.com/ye-kyaw-thu/myPOS)  

## Install LIBLINEAR

In [9]:
%cd /home/ye/aif2/svm

/home/ye/aif2/svm


In [10]:
!git clone https://github.com/cjlin1/liblinear.git

Cloning into 'liblinear'...
remote: Enumerating objects: 1730, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 1730 (delta 139), reused 109 (delta 109), pack-reused 1555 (from 2)
Receiving objects: 100% (1730/1730), 4.75 MiB | 18.15 MiB/s, done.
Resolving deltas: 100% (1151/1151), done.


In [11]:
%cd liblinear/

/home/ye/aif2/svm/liblinear


In [12]:
!make

g++ -Wall -Wconversion -O3 -fPIC -c -o newton.o newton.cpp
g++ -Wall -Wconversion -O3 -fPIC -c -o linear.o linear.cpp
make -C blas OPTFLAGS='-Wall -Wconversion -O3 -fPIC' CC='cc';
make[1]: Entering directory '/home/ye/aif2/svm/liblinear/blas'
cc -Wall -Wconversion -O3 -fPIC -c dnrm2.c
cc -Wall -Wconversion -O3 -fPIC -c daxpy.c
cc -Wall -Wconversion -O3 -fPIC -c ddot.c
cc -Wall -Wconversion -O3 -fPIC -c dscal.c
ar rcv blas.a dnrm2.o daxpy.o ddot.o dscal.o
a - dnrm2.o
a - daxpy.o
a - ddot.o
a - dscal.o
ranlib blas.a
make[1]: Leaving directory '/home/ye/aif2/svm/liblinear/blas'
g++ -Wall -Wconversion -O3 -fPIC -o train train.c newton.o linear.o blas/blas.a
g++ -Wall -Wconversion -O3 -fPIC -o predict predict.c newton.o linear.o blas/blas.a


### Check

make လုပ်ပြီး installation လုပ်တာ အဆင်ပြေရင် train နဲ့ predict ဆိုတဲ့ ပရိုဂရမ်နှစ်ခု ရလာလိမ့်မယ်။

In [14]:
!ls train predict

predict  train


## Data Preparation

myPOS dataset ကိုပဲ သုံးပြီးတော့ SVMs နဲ့ မြန်မာစာအတွက် POS tagging ကို လုပ်ကြည့်ကြရအောင်။ အဲဒီအတွက် [myPOS](https://github.com/ye-kyaw-thu/myPOS) ဒေတာကို ကိုယ့်စက်ထဲကို download လုပ်ပါ။  

In [15]:
%pwd

'/home/ye/aif2/svm/liblinear'

In [17]:
%cd ..

/home/ye/aif2/svm


In [18]:
!mkdir data

In [19]:
%cd data

/home/ye/aif2/svm/data


**ပထမဆုံး training dataset ဖြစ်တဲ့ myPOS version3 ကို download လုပ်မယ်။**

In [20]:
!wget https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt

--2026-07-28 09:12:26--  https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt [following]
--2026-07-28 09:12:27--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9581544 (9.1M) [application/octet-stream]
Saving to: ‘mypos-ver.3.0.shuf.nopipe.txt’

mypos-ver.3.0.shuf. 100%[========

**test dataset ကိုလည်း download လုပ်ကြရအောင်။**

In [21]:
!wget https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt

--2026-07-28 09:14:54--  https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt [following]
--2026-07-28 09:14:54--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 229758 (224K) [text/plain]
Saving to: ‘otest.1k.nopipe.txt’

otest.1k.nopipe.txt 100%[===================>] 224.37K  --.-KB/s    in 0.007s  

2026-07

### Check myPOS (Version 3.0) Dataset

In [22]:
!wc *

  43196  564517 9581544 mypos-ver.3.0.shuf.nopipe.txt
   1000   13468  229758 otest.1k.nopipe.txt
  44196  577985 9811302 total


In [23]:
!head mypos-ver.3.0.shuf.nopipe.txt

၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc
လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျော်စွာ/ppm သတ်မှတ်/v ထား/part သည့်/part အလုပ်/n အားလပ်ရက်/n များ/part ပါဝင်/v သည့်/part အနားယူခွင့်/n နှင့်/conj အားလပ်ခွင့်/n ခံစားပိုင်ခွင့်/n ရှိ/v သည်/ppm ။/punc
ဤ/adj နည်း/n ကို/ppm စစ်ယူ/v သော/part နည်း/n ဟု/part ခေါ်/v သည်/ppm ။/punc
စာပြန်ပွဲ/n ဆို/v တာ/part က/ppm အာဂုံဆောင်/v အလွတ်ကျက်/v ထား/part တဲ့/part ပိဋကတ်သုံးပုံ/n စာပေ/n တွေ/part ကို/ppm စာစစ်/v သံဃာတော်ကြီး/n တွေ/part ရဲ့/ppm ရှေ့/n မှာ/ppm အလွတ်/adv ပြန်/v ပြီး/part ရွတ်ပြ/v ရ/part တာ/part ပေါ့/part ။/punc
ဒီ/pron မှာ/ppm ကျွန်တော့်/pron သက်သေခံကတ်/n ပါ/part ။/punc
၂ဝ/num ရာစု/n မြန်မာ့/n သမိုင်း/n သန်းဝင်းလှိုင်/n ၊/punc ၂ဝဝ၉/num ခု/part ၊/punc မေ/n လ/n ၊/punc ကံကော်ဝတ်ရည်/n စာပေ/n ။/punc
ကျွန်တော်/pron မျက်မှန်/n တစ်/tn လက်/part လုပ်/v ချင်/part ပါ/p

In [24]:
!head otest.1k.nopipe.txt

တစ်/tn ကိုက်/n ကို/ppm ဝမ်/n ခုနှစ်ထောင်/tn ပါ/part ။/punc
မနှစ်/n က/ppm သူ/pron ကျွန်မ/pron ကို/ppm သင်/v ပေး/part တယ်/ppm ။/punc
ကျွန်တော့်/pron ခုံ/n သွား/v ရှာ/v မလို့/part ။/punc
အတန်း/n စ/v တာ/part ကြာ/v ပြီ/ppm လား/part ။/punc
ဆေး/n နည်းနည်း/adv စား/v လိုက်/part ၊/punc သုံး/tn လေး/tn ရက်/n လောက်/part အနားယူ/v လိုက်/part ရင်/conj ပျောက်/v သွား/part မှာ/ppm ပါ/part ။/punc
အေးချမ်း/v မှု/part နဲ့/conj စည်းကမ်း/n ကို/ppm တည်မြဲ/v အောင်/part ထိန်းသိမ်း/v သည်/ppm ။/punc
ဇွန်း/n ကို/ppm လိုအပ်/v တယ်/ppm ။/punc
ဘွဲ့/n ရ/v ရင်/conj ဘာ/n လုပ်/v မ/part လို့/part လဲ/part ။/punc
ကျွန်တော်/pron ချောင်းဆိုး/v ခြင်း/part အတွက်/ppm တစ်/tn ခု/part ခု/part လို/v ချင်/part တယ်/ppm ။/punc
အသီးအနှံ/n တို့/part မှ/ppm လွဲ/v လျှင်/conj လူ/n တို့/part ၏/ppm အဓိက/n အစားအစာ/n မှာ/ppm ငါး/n ဖြစ်/v သည်/ppm ။/punc


## Python and Shell Scripts

Preprocessing, training/testing နဲ့ evaluation အလုပ်တွေအတွက် လိုအပ်တဲ့ Python code, shell script တွေကို ရေးပြထားပါတယ်။  

In [27]:
%cd /home/ye/aif2/svm

/home/ye/aif2/svm


In [28]:
!ls *.py

01_preprocess.py  02_featurize.py  04_evaluate.py


In [29]:
!ls *.sh

03_run_liblinear.sh


In [30]:
!mkdir work

In [59]:
!time python3 ./01_preprocess.py data/mypos-ver.3.0.shuf.nopipe.txt ./work/train.conll

[preprocess] data/mypos-ver.3.0.shuf.nopipe.txt → ./work/train.conll

real	0m0.291s
user	0m0.268s
sys	0m0.016s


In [60]:
!head -n 50 ./work/train.conll

၁၉၆၂	num
ခုနှစ်	n
ခန့်မှန်း	v
သန်းခေါင်စာရင်း	n
အရ	ppm
လူဦးရေ	n
၁၁၅၉၃၁	num
ယောက်	part
ရှိ	v
သည်	ppm
။	punc

လူ	n
တိုင်း	part
တွင်	ppm
သင့်မြတ်	v
လျော်ကန်	v
စွာ	part
ကန့်သတ်	v
ထား	part
သည့်	part
အလုပ်	n
လုပ်	v
ချိန်	n
အပြင်	conj
၊	punc
လစာ	n
နှင့်တကွ	conj
အခါ	n
ကာလ	n
အားလျော်စွာ	ppm
သတ်မှတ်	v
ထား	part
သည့်	part
အလုပ်	n
အားလပ်ရက်	n
များ	part
ပါဝင်	v
သည့်	part
အနားယူခွင့်	n
နှင့်	conj
အားလပ်ခွင့်	n
ခံစားပိုင်ခွင့်	n
ရှိ	v
သည်	ppm
။	punc

ဤ	adj
နည်း	n
ကို	ppm


**test data အတွက်လည်း ကော်လံပုံစံ data format ကို ပြင်ဆင်ကြရအောင်။**

In [61]:
!time python3 ./01_preprocess.py data/otest.1k.nopipe.txt ./work/test.conll

[preprocess] data/otest.1k.nopipe.txt → ./work/test.conll

real	0m0.029s
user	0m0.020s
sys	0m0.009s


In [62]:
!head -n 50 ./work/test.conll

တစ်	tn
ကိုက်	n
ကို	ppm
ဝမ်	n
ခုနှစ်ထောင်	tn
ပါ	part
။	punc

မနှစ်	n
က	ppm
သူ	pron
ကျွန်မ	pron
ကို	ppm
သင်	v
ပေး	part
တယ်	ppm
။	punc

ကျွန်တော့်	pron
ခုံ	n
သွား	v
ရှာ	v
မလို့	part
။	punc

အတန်း	n
စ	v
တာ	part
ကြာ	v
ပြီ	ppm
လား	part
။	punc

ဆေး	n
နည်းနည်း	adv
စား	v
လိုက်	part
၊	punc
သုံး	tn
လေး	tn
ရက်	n
လောက်	part
အနားယူ	v
လိုက်	part
ရင်	conj
ပျောက်	v
သွား	part
မှာ	ppm
ပါ	part
။	punc


## Code: 01_preprocess.py

Line format ကနေ column format ပြောင်းပေးတဲ့ Python code ကိုလည်း လေ့လာပါ။ လက်တွေ့ မြန်မာစာ၊ ခမာစာ၊ ထိုင်းစာ၊ အင်္ဂလိပ်စာ စတဲ့ ဘာသာစကားတွေနဲ့ မော်ဒယ်ဆောက်မယ် ဆိုရင် ဒါမျိုး အလုပ်တွေက အကြိမ်ကြိမ်အခါခါ လုပ်ကြရမှာမို့... 

In [63]:
from IPython.display import Code

# Display the script with Python syntax highlighting
Code(filename='01_preprocess.py', language='python')

#!/usr/bin/env python3
"""
Convert myPOS format  →  CoNLL format (one token per line, word \t tag).
Handles compound tags like  ဘောဂ/n|ဗေဒ/n  by splitting on '|'.
"""

import sys

def parse_line(line):
    tokens = []
    for tok in line.strip().split():
        # Split compound words on '|':  ဘောဂ/n|ဗေဒ/n  →  ဘောဂ/n  +  ဗေဒ/n
        for part in tok.split('|'):
            if '/' not in part:
                continue
            idx = part.rfind('/')          # rfind: word may itself contain '/'
            word = part[:idx]
            tag  = part[idx + 1:]
            if word and tag:
                tokens.append((word, tag))
    return tokens

def main():
    inp, outp = sys.argv[1], sys.argv[2]
    with open(inp, encoding='utf-8') as f, \
         open(outp, 'w', encoding='utf-8') as out:
        for line in f:
            toks = parse_line(line)
            if not toks:
                continue
            for w, t in toks:
                out.write(f'{w}\t{t}\n')
            out.write('\n')          # blank line = sentence boundary
    print(f'[preprocess] {inp} → {outp}')

if __name__ == '__main__':
    main()

## Extract features → LIBLINEAR (libsvm) format

In [64]:
!time python3 ./02_featurize.py train  ./work/train.conll  ./work/train.svm  ./work/vocab.pkl

[featurize] vocab size: 331719 feats, 15 labels
[featurize] train: ./work/train.conll → ./work/train.svm

real	0m5.044s
user	0m4.943s
sys	0m0.086s


In [65]:
!head ./work/train.svm

1 1:1 2:1 3:1 4:1 5:1 6:1 7:1 8:1 9:1 10:1 11:1
2 12:1 13:1 14:1 15:1 16:1 17:1 18:1 19:1 20:1 21:1 22:1 23:1 24:1
3 13:1 19:1 25:1 26:1 27:1 28:1 29:1 30:1 31:1 32:1 33:1 34:1 35:1 36:1
2 19:1 26:1 28:1 37:1 38:1 39:1 40:1 41:1 42:1 43:1 44:1 45:1 46:1 47:1
4 19:1 48:1 49:1 50:1 51:1 52:1 53:1 54:1 55:1 56:1 57:1 58:1
2 19:1 20:1 59:1 60:1 61:1 62:1 63:1 64:1 65:1 66:1 67:1 68:1 69:1 70:1
1 2:1 8:1 20:1 71:1 72:1 73:1 74:1 75:1 76:1 77:1 78:1 79:1 80:1 81:1
5 14:1 19:1 82:1 83:1 84:1 85:1 86:1 87:1 88:1 89:1 90:1 91:1 92:1 93:1
3 19:1 94:1 95:1 96:1 97:1 98:1 99:1 100:1 101:1 102:1 103:1 104:1 105:1 106:1
4 14:1 19:1 38:1 101:1 107:1 108:1 109:1 110:1 111:1 112:1 113:1 114:1 115:1


In [66]:
!tail ./work/train.svm

3 19:1 38:1 53:1 96:1 105:1 750:1 1993:1 1994:1 1995:1 9028:1 143681:1
4 19:1 61:1 88:1 114:1 333:1 334:1 759:1 1997:1 2614:1 4118:1 9030:1 9031:1 9032:1
6 19:1 116:1 117:1 118:1 119:1 1999:1 9033:1
2 19:1 183:1 462:1 501:1 503:1 895:1 1225:1 1887:1 21403:1 21404:1 29275:1
4 19:1 119:1 179:1 465:1 466:1 21407:1 29276:1 135291:1 331716:1
9 19:1 49:1 150:1 168:1 210:1 475:1 480:1 1426:1 1561:1 13833:1 21410:1 29278:1 135293:1 331717:1
3 14:1 19:1 31:1 92:1 211:1 333:1 483:1 583:1 1566:1 2833:1 9770:1 29279:1 135295:1 331718:1
5 19:1 53:1 61:1 104:1 105:1 264:1 1568:1 1569:1 1570:1 29281:1 135296:1 331719:1
4 14:1 19:1 38:1 101:1 107:1 108:1 109:1 110:1 111:1 114:1 1574:1 1576:1 135298:1
6 19:1 116:1 117:1 118:1 119:1 120:1 1577:1


In [67]:
!wc ./work/train.svm

  564517  7649458 46332379 ./work/train.svm


In [68]:
!time python3 ./02_featurize.py test  ./work/test.conll  ./work/test.svm  ./work/vocab.pkl

[featurize] test: ./work/test.conll → ./work/test.svm

real	0m0.196s
user	0m0.169s
sys	0m0.027s


In [69]:
!head ./work/test.svm

11 14:1 16:1 19:1 101:1 128:1 378:1 754:1 755:1 756:1 757:1 20304:1
2 14:1 19:1 85:1 88:1 179:1 384:1 389:1 391:1 766:1 961:1 1823:1 9876:1 20306:1
4 19:1 101:1 179:1 387:1 388:1 389:1 390:1 391:1 392:1 771:1 9878:1 20307:1 326391:1 326392:1
2 14:1 19:1 101:1 403:1 641:1 1663:1 1726:1 2013:1 2015:1 9880:1 9881:1 20309:1 326393:1 326394:1
11 13:1 14:1 15:1 17:1 19:1 42:1 105:1 137:1 414:1 474:1 648:1 9884:1 46087:1 326395:1
5 19:1 53:1 114:1 276:1 333:1 334:1 650:1 651:1 9887:1 326396:1 326397:1
6 19:1 116:1 117:1 118:1 119:1 655:1 326398:1
2 14:1 16:1 18:1 19:1 88:1 325:1 462:1 1201:1 3695:1 5427:1 5428:1
4 19:1 119:1 179:1 465:1 466:1 1204:1 5430:1 5989:1 326399:1
10 19:1 38:1 53:1 123:1 378:1 475:1 1206:1 1207:1 1208:1 5438:1 5992:1 26722:1


In [70]:
!tail ./work/test.svm

3 19:1 38:1 53:1 96:1 105:1 750:1 1993:1 1994:1 1995:1 9028:1 143681:1
4 19:1 61:1 88:1 114:1 333:1 334:1 759:1 1997:1 2614:1 4118:1 9030:1 9031:1 9032:1
6 19:1 116:1 117:1 118:1 119:1 1999:1 9033:1
2 19:1 183:1 462:1 501:1 503:1 895:1 1225:1 1887:1 21403:1 21404:1 29275:1
4 19:1 119:1 179:1 465:1 466:1 21407:1 29276:1 135291:1 331716:1
9 19:1 49:1 150:1 168:1 210:1 475:1 480:1 1426:1 1561:1 13833:1 21410:1 29278:1 135293:1 331717:1
3 14:1 19:1 31:1 92:1 211:1 333:1 483:1 583:1 1566:1 2833:1 9770:1 29279:1 135295:1 331718:1
5 19:1 53:1 61:1 104:1 105:1 264:1 1568:1 1569:1 1570:1 29281:1 135296:1 331719:1
4 14:1 19:1 38:1 101:1 107:1 108:1 109:1 110:1 111:1 114:1 1574:1 1576:1 135298:1
6 19:1 116:1 117:1 118:1 119:1 120:1 1577:1


In [71]:
!wc ./work/test.svm

  13468  183055 1110702 ./work/test.svm


## Understanding the .svm Feature File Format  

The file train.svm is in Sparse LibSVM/LIBLINEAR format. Because NLP features are mostly binary (1 or 0) and we have 331,719 possible features, writing out 331,719 numbers for every single word would make the file huge and slow to read. Instead, we only write the features that are active (value = 1) for that specific word.  

The format of training and testing data file is:   

```
<label> <index1>:<value1> <index2>:<value2> ...  
```

```
1 1:1 2:1 3:1 4:1 5:1 6:1 7:1 8:1 9:1 10:1 11:1
```

Here is exactly what this means:  

- `1` (at the very beginning) = The Label (POS Tag ID). In our `Vocab` class, we mapped the tags to numbers starting from 1. So `1` might mean `abb` or `n` (depending on how they appeared in the data).
- `1:1` = Feature ID `1` has a value of `1`.
- `2:1` = Feature ID `2` has a value of `1`.
- `11:1` = Feature ID `11` has a value of `1`.

## Code Reading Time

**Question 1:** `02_featurize.py` code ကို run တဲ့အခါမှာ ဘာကြောင့် "train" နဲ့ "test" ကိုခွဲထားရတာလဲ ဆိုတာကို code ကိုဖတ်ရင်း စဉ်းစားကြည့်ပါ။  
**Question 2:** `ids = sorted({vocab.get_feat(f) for f in feats if vocab.get_feat(f) > 0})` လိုင်းကို ဖတ်ကြည့်ပါ။ တကယ်လို့ Out-Of-Vocabulary (OOV) or Unseen feature case ဖြစ်လာရင် လက်ရှိ code က ဘယ်လို ဖြေရှင်းမှာလဲ။  

## Code: 02_featurize.py

Feature ဆွဲထုတ်တဲ့ Python code ကိုလည်း self study လုပ်ကြပါ။  

In [72]:
from IPython.display import Code

# Display the script with Python syntax highlighting
Code(filename='/home/ye/aif2/svm/02_featurize.py', language='python')

#!/usr/bin/env python3
"""
Extract word-level features and write libsvm-format files for LIBLINEAR.
Labels are 1-indexed (1 to K) as required by LIBLINEAR.
"""

import sys
import pickle

def read_conll(path):
    sents, cur = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if cur:
                    sents.append(cur)
                    cur = []
            else:
                w, t = line.split('\t')
                cur.append((w, t))
    if cur:
        sents.append(cur)
    return sents

def extract_features(sent, i):
    """Return list of string features for token at position i."""
    word = sent[i][0]
    feats = []
    feats.append(f'w={word}')
    for n in (1, 2, 3):
        if len(word) >= n:
            feats.append(f'p{n}={word[:n]}')
            feats.append(f's{n}={word[-n:]}')
    feats.append(f'has_digit={any(c.isdigit() for c in word)}')
    feats.append(f'len={min(len(word), 10)}')
    
    # context words
    if i > 0:
        feats.append(f'w-1={sent[i-1][0]}')
    if i > 1:
        feats.append(f'w-2={sent[i-2][0]}')
    if i < len(sent) - 1:
        feats.append(f'w+1={sent[i+1][0]}')
    if i < len(sent) - 2:
        feats.append(f'w+2={sent[i+2][0]}')
        
    # surrounding word bigram
    if 0 < i < len(sent) - 1:
        feats.append(f'w-1_w+1={sent[i-1][0]}|{sent[i+1][0]}')
    return feats

class Vocab:
    def __init__(self):
        self.feat2id  = {}     # feature string → int (1-indexed)
        self.label2id = {}     # tag string → int (1-indexed)
        self.id2label = []     # int → tag string (0-indexed list)
        
    def add_feat(self, f):
        if f not in self.feat2id:
            self.feat2id[f] = len(self.feat2id) + 1
            
    def get_feat(self, f):
        return self.feat2id.get(f, -1)
        
    def add_label(self, l):
        if l not in self.label2id:
            self.label2id[l] = len(self.id2label) + 1  # 1-indexed label
            self.id2label.append(l)
            
    def get_label(self, l):
        return self.label2id.get(l, -1)

def main():
    mode        = sys.argv[1]   # 'train' or 'test'
    conll_path  = sys.argv[2]
    out_path    = sys.argv[3]
    vocab_path  = sys.argv[4]

    sents = read_conll(conll_path)

    if mode == 'train':
        vocab = Vocab()
        for sent in sents:
            for i in range(len(sent)):
                for f in extract_features(sent, i):
                    vocab.add_feat(f)
                vocab.add_label(sent[i][1])
        with open(vocab_path, 'wb') as vf:
            pickle.dump(vocab, vf)
        print(f'[featurize] vocab size: {len(vocab.feat2id)} feats, '
              f'{len(vocab.id2label)} labels')
    else:
        with open(vocab_path, 'rb') as vf:
            vocab = pickle.load(vf)

    with open(out_path, 'w', encoding='utf-8') as out:
        for sent in sents:
            for i in range(len(sent)):
                feats = extract_features(sent, i)
                # Use a set to remove duplicate feature IDs, then sort for libsvm format
                ids = sorted({vocab.get_feat(f) for f in feats if vocab.get_feat(f) > 0})
                
                # ALWAYS output the true label (1-indexed) 
                # Even for test set, so liblinear predict can calculate accuracy automatically
                label = vocab.get_label(sent[i][1])
                feat_str = ' '.join(f'{fid}:1' for fid in ids)
                
                # Write label followed by space-separated features
                out.write(f'{label} {feat_str}\n')
                
    print(f'[featurize] {mode}: {conll_path} → {out_path}')

if __name__ == '__main__':
    main()

**LIBSVM က default ထားတဲ့ ဖိုင်နာမည်နဲ့ပဲ သွားချင်လို့ myPOS ဒေတာတွေကို train.txt, test.txt အဖြစ် ကော်ပီကူးလိုက်တယ်။ အဲဒီလို မလုပ်ချင်ရင်တော့ shell script: 03_run_liblinear.sh ဖိုင်မှာ ဝင်ပြင်ပါ။**

In [47]:
%cd /home/ye/aif2/svm/data

/home/ye/aif2/svm/data


In [48]:
!cp mypos-ver.3.0.shuf.nopipe.txt train.txt 

In [49]:
!cp otest.1k.nopipe.txt test.txt

In [50]:
!ls *

mypos-ver.3.0.shuf.nopipe.txt  otest.1k.nopipe.txt  test.txt  train.txt


## Train, predict, evaluate

အထက်မှာတော့ preprocessing လုပ်တဲ့အပိုင်း၊ feature ဆွဲထုတ်တဲ့အပိုင်းတွေကို မြင်သာအောင် သပ်သပ်စီ run ပြခဲ့ပေမဲ့ တကယ်တမ်း အလုပ်လုပ်တဲ့အခါမှာတော့ shell script ရေးထားပြီး preprocessing ကနေ နောက်ဆုံး  evaluation လုပ်တဲ့အထိ လုပ်လေ့ရှိကြပါတယ်။  

အောက်ပါ shell script ကို သုံးပြီး SVMs model ဆောက်မယ် (i.e. training) ပြီးရင် test data နဲ့ testing သို့မဟုတ် prediction လုပ်မယ်။ ပြီးရင် ရလဒ်ဘယ်လောက် ကောင်းသလဲ ဆိုတာကို evaluation လုပ်ကြည့်မယ်။  

In [73]:
from IPython.display import Code

# Display the script with Python syntax highlighting
Code(filename='/home/ye/aif2/svm/03_run_liblinear.sh', language='bash')

#!/bin/bash
# ============================================================
#  SVM POS Tagger — LIBLINEAR pipeline
#  Usage: bash 03_run_liblinear.sh /path/to/liblinear
# ============================================================

set -e

DATA_DIR=${DATA_DIR:-./data}       # folder with train.txt / test.txt
WORK_DIR=${WORK_DIR:-./work}
LIBLINEAR_HOME=${1:-./liblinear}   # path to liblinear source dir

TRAIN_BIN="$LIBLINEAR_HOME/train"
PRED_BIN="$LIBLINEAR_HOME/predict"

mkdir -p "$WORK_DIR"

# ---- Step 1: Preprocess myPOS → CoNLL ----
echo "=== [1/5] Preprocessing ==="
python3 01_preprocess.py "$DATA_DIR/train.txt" "$WORK_DIR/train.conll"
python3 01_preprocess.py "$DATA_DIR/test.txt"  "$WORK_DIR/test.conll"

# ---- Step 2: Featurize → libsvm format ----
echo "=== [2/5] Feature extraction ==="
python3 02_featurize.py train "$WORK_DIR/train.conll" \
                          "$WORK_DIR/train.svm"  "$WORK_DIR/vocab.pkl"
python3 02_featurize.py test  "$WORK_DIR/test.conll"  \
                          "$WORK_DIR/test.svm"   "$WORK_DIR/vocab.pkl"

# ---- Step 3: Train LIBLINEAR ----
#   -s 2  : L2-regularized L2-loss SVM (dual)
#   -c 1.0: regularization cost
echo "=== [3/5] Training LIBLINEAR ==="
"$TRAIN_BIN" -s 2 -c 1.0 -e 0.001 "$WORK_DIR/train.svm" "$WORK_DIR/model.bin"

# ---- Step 4: Predict ----
echo "=== [4/5] Predicting ==="
"$PRED_BIN" "$WORK_DIR/test.svm" "$WORK_DIR/model.bin" \
            "$WORK_DIR/test.pred" 2>&1 | tee "$WORK_DIR/predict.log"

# ---- Step 5: Evaluate ----
echo "=== [5/5] Evaluation ==="
python3 04_evaluate.py "$WORK_DIR/test.conll" \
                       "$WORK_DIR/test.pred"   \
                       "$WORK_DIR/vocab.pkl"

echo "=== Done ==="

In [74]:
%cd /home/ye/aif2/svm/

/home/ye/aif2/svm


In [57]:
!chmod +x ./03_run_liblinear.sh

!time ./03_run_liblinear.sh 

## What Do the logs mean?

အထက်ပါ Training log ကနေ လေ့လာရသလောက် learning curve က အဆင်ပြေပါတယ်။
LIBLINEAR က SVM model အတွက် optimal weight တွေ ရဖို့အတွက် သုံးသွားတဲ့ optimization algorithm ကို Trust Region Newton Method (TRON) လို့ခေါ်ပါတယ်။

```
init f 5.645e+05 |g| 1.331e+06
iter  1 f 2.816e+03 |g| 4.453e+04 CG   2 step_size 1.00e+00 
iter  2 f 7.823e+02 |g| 1.384e+04 CG   2 step_size 1.00e+00 
...
iter 15 f 4.495e+01 |g| 8.018e+00 CG   6 step_size 1.00e+00 
```

- **init f:** The initial objective function value (the SVM's initial error/loss). It starts very high (5.645e+05).
- **iter:** The iteration number. Notice it only takes 15 iterations to converge. This is extremely fast.
- **f:** The current objective function value. You want this to drop. It drops from 5.6e+05 down to 44.95 very quickly. This means the model is learning the patterns in the data extremely well.
- **|g|:** The norm of the gradient. This tells you how steep the error curve is. It drops from 1.3e+06 to 8.0. When this number gets very close to 0, it means the model has reached the bottom of the error curve (converged).
- **CG:** Conjugate Gradient iterations. It's an internal math step to find the search direction.
- **step_size:** How big of a jump the algorithm takes to update the weights.


## Why are there 15 blocks of these logs?

LIBLINEAR က POS tag 15 ခုအတွက် One-vs-Rest (OVR) လို့ခေါ်တဲ့ multi-class classifier အနေနဲ့ train လုပ်ပါတယ်။ ဆိုလိုတာက POS tag တစ်ခုစီအတွက် binary SVM မော်ဒယ် တစ်ခုစီဆောက် ဆောက်သွားတဲ့ ပုံစံပါ။ စုစုပေါင်း မော်ဒယ် ၁၅ ခုပါ။  

## pkl2txt Conversion

In [76]:
%pwd

'/home/ye/aif2/svm'

In [77]:
!python ./pkl2txt.py --help

usage: pkl2txt.py [-h] -i INPUT [-o OUTPUT] [--sep SEP]

Convert a pickle (.pkl) file to plain text format.

options:
  -h, --help            show this help message and exit
  -i INPUT, --input INPUT
                        Path to the input .pkl file. (default: None)
  -o OUTPUT, --output OUTPUT
                        Path to the output text file. If not given, prints to
                        stdout. (default: None)
  --sep SEP             Separator between key and value for dict-like objects
                        (default: TAB). (default: )


In [80]:
!time python ./pkl2txt.py --input ./work/vocab.pkl --output ./work/vocab.txt

[INFO] Output written to: ./work/vocab.txt

real	0m0.170s
user	0m0.135s
sys	0m0.035s


In [81]:
!head -n 100 ./work/vocab.txt

# === Feat2ID (feature string -> int) ===
w=၁၉၆၂	1
p1=၁	2
s1=၂	3
p2=၁၉	4
s2=၆၂	5
p3=၁၉၆	6
s3=၉၆၂	7
has_digit=True	8
len=4	9
w+1=ခုနှစ်	10
w+2=ခန့်မှန်း	11
w=ခုနှစ်	12
p1=ခ	13
s1=်	14
p2=ခု	15
s2=စ်	16
p3=ခုန	17
s3=ှစ်	18
has_digit=False	19
len=6	20
w-1=၁၉၆၂	21
w+1=ခန့်မှန်း	22
w+2=သန်းခေါင်စာရင်း	23
w-1_w+1=၁၉၆၂|ခန့်မှန်း	24
w=ခန့်မှန်း	25
s1=း	26
p2=ခန	27
s2=်း	28
p3=ခန်	29
s3=န်း	30
len=9	31
w-1=ခုနှစ်	32
w-2=၁၉၆၂	33
w+1=သန်းခေါင်စာရင်း	34
w+2=အရ	35
w-1_w+1=ခုနှစ်|သန်းခေါင်စာရင်း	36
w=သန်းခေါင်စာရင်း	37
p1=သ	38
p2=သန	39
p3=သန်	40
s3=င်း	41
len=10	42
w-1=ခန့်မှန်း	43
w-2=ခုနှစ်	44
w+1=အရ	45
w+2=လူဦးရေ	46
w-1_w+1=ခန့်မှန်း|အရ	47
w=အရ	48
p1=အ	49
s1=ရ	50
p2=အရ	51
s2=အရ	52
len=2	53
w-1=သန်းခေါင်စာရင်း	54
w-2=ခန့်မှန်း	55
w+1=လူဦးရေ	56
w+2=၁၁၅၉၃၁	57
w-1_w+1=သန်းခေါင်စာရင်း|လူဦးရေ	58
w=လူဦးရေ	59
p1=လ	60
s1=ေ	61
p2=လူ	62
s2=ရေ	63
p3=လူဦ	64
s3=းရေ	65
w-1=အရ	66
w-2=သန်းခေါင်စာရင်း	67
w+1=၁၁၅၉၃၁	68
w+2=ယောက်	69
w-1_w+1=အရ|၁၁၅၉၃၁	70
w=၁၁၅၉၃၁	71
s1=၁	72
p2=၁၁	73
s2=၃၁	74
p3=၁၁၅	75
s3=၉၃၁	76
w-1=လ

## Training Parameters of LIBLINEAR

In [82]:
!./liblinear/train -h

Usage: train [options] training_set_file [model_file]
options:
-s type : set type of solver (default 1)
  for multi-class classification
        0 -- L2-regularized logistic regression (primal)
        1 -- L2-regularized L2-loss support vector classification (dual)
        2 -- L2-regularized L2-loss support vector classification (primal)
        3 -- L2-regularized L1-loss support vector classification (dual)
        4 -- support vector classification by Crammer and Singer
        5 -- L1-regularized L2-loss support vector classification
        6 -- L1-regularized logistic regression
        7 -- L2-regularized logistic regression (dual)
  for regression
       11 -- L2-regularized L2-loss support vector regression (primal)
       12 -- L2-regularized L2-loss support vector regression (dual)
       13 -- L2-regularized L1-loss support vector regression (dual)
  for outlier detection
       21 -- one-class support vector machine (dual)
-c cost : set the parameter C (default 1)
-p eps

## Summary of LIBLINEAR Parameters  


* **`-c` (Cost / Regularization parameter):**
  Controls the trade-off between training accuracy and model simplicity.
  * **Higher C (e.g., 10, 100):** Tries harder to classify every training example correctly. Can lead to overfitting.
  * **Lower C (e.g., 0.1, 0.01):** Allows more errors on the training set but creates a wider margin, potentially generalizing better to unseen data.

* **`-s` (Solver type):**
  * `-s 2`: L2-loss SVM (Primal). Good for general use.
  * `-s 1`: L2-loss SVM (Dual). Often faster when the number of features is much larger than the number of samples.
  * `-s 0`: Logistic Regression. Outputs probabilities (if you use `-b 1` in predict) and sometimes handles overlapping classes better.
  * `-s 5`: L1-regularized. Forces the model to use fewer features (sparse model), acting as a built-in feature selector.

* **`-B` (Bias):**
  Adding `-B 1` adds an extra feature to every vector. This helps the model shift the decision boundary without relying on the data always being centered around zero.

* **`-v` (Cross Validation):**
  Using `-v 5` does 5-fold cross-validation on the training set. It is the correct way to tune `C` without touching the test set.


## Understanding on Training/Testing Command Line Parameters

### 'train' Command Parameters

**`-s type` : The Solver (The Algorithm)** က LIBLINEAR ကို ဘယ် "mathematical algorithm" ကို သုံးမယ် ဆိုတာကို ညွှန်ကြားပေးပါတယ်။  Multi-class classification အတွက် ဆိုရင် ဆရာတို့က `0` ကနေ `7` အထိ ပေးလို့ ရပါတယ်။  

*   **Option 1 & 2 (L2-loss SVM):** Standard Support Vector Machines အနေနဲ့ သုံးလေ့ရှိပါတယ်။  လိုင်း (သို့) ဘော်ဒါ သတ်မှတ်တဲ့အခါမှာ maximum margin ရသလောက် ယူပြီး သွားပါတယ်။  
    *   `1` is **dual**. ကိုယ့်ရဲ့ ဒေတာမှာ feature တွေက တအားများနေပြီး sample ဒေတာက နည်းနေတဲ့ အခါမျိုးမှာ ကောင်းပါတယ်။ 
    *   `2` is **primal**. ကိုယ့်ရဲ့ ဒေတာမှာ sample ဒေတာက က တအားများနေပြီး၊ feature တွေက နည်းနေတဲ့ အခြေအနေမျိုးမှာ ကောင်းပါတယ်။  
*   **Option 0 & 7 (Logistic Regression):** Hard margin မလုပ်ပဲ probability တန်ဖိုးတွေကို ထုတ်ပေးပါလိမ့်မယ်။ ကိုယ်မော်ဒယ်က စာကြောင်းတွေကို POS tagging လုပ်သွားတဲ့အခါမှာ ဘယ်လောက် (သို့) ဘယ်လို ယုံယုံကြည်ကြည်နဲ့ (i.e. confident) လုပ်သွားသလဲ ဆိုတာကို သိချင်တဲ့အခါမျိုးမှာ သုံးပါတယ်။ ဥပမာ ဒီစာလုံးက "နမ်" ဖြစ်ဖို့ 80% သေချာပြီး "ကြိယာ" ဖြစ်နိုင်ခြေက 15% ပဲ ရှိတယ် ဆိုတာမျိုး။ ဒီလို option နဲ့ run တဲ့အခါမှာ အတွေ့အကြုံအရတော့ ပုံမှန်ထားလေ့ရှိတဲ့ `1` or `2` ထက် accuracy ရလဒ်က အနည်းငယ် လျော့လေ့ ရှိပါတယ်။
*   **Option 3 (L1-loss SVM):** Standard SVM နဲ့ ဆင်ပါတယ်။ ဒါပေမဲ့ သူက error တွေကို penalize လုပ်တဲ့ ပုံစံက Standard SVM မှာ လုပ်တာနဲ့ မတူပါဘူး။ ကိုယ့်ဒေတာထဲမှာ outlier (စုန်းပြူး) တွေ အများကြီးရှိနေရင်တော့ `L1-loss SVM` ကို သုံးကြည့်ပါ။ Outlier ဆိုတာက လေဘယ်မှားထိုးတာမျိုးကို ဆိုလိုတာပါ။  
*   **Option 5 & 6 (L1-regularized):** ဒီ option ကတော့ အသုံးမဝင်နိုင်တဲ့ feature တွေကို ဂရုမစိုက်တော့ပဲ training လုပ်သွားတဲ့ ပုံစံပါ။ နည်းပညာ စကားနဲ့ ပြောရရင်တော့ weight vector sparse လုပ်ပါတယ်။ အဲဒီလိုလုပ်ရင် training process အတွက် လိုအပ်တဲ့ feature တွေကို ရွေးချယ်တဲ့နေရာမှာတော့ ကောင်းပါတယ်။ ဒါပေမဲ့ သူက training time ကိုတော့ နှေးစေလိမ့်မယ်။  

#### **`-c cost` : The Regularization Parameter (Crucial for Tuning)**

ဒီ `-c` parameter ကတော့ accuracy ကောင်းဖို့အတွက် အရေးအကြီးဆုံးလို့ ပြောလို့ ရပါတယ်။ ဒေတာတွေကို classification လုပ်တဲ့အပိုင်းနဲ့ လိုင်းနေရာချတာကို ချောချောမွေ့မွေ့ မှန်မှန်ကန်ကန် ချပေးနိုင်တဲ့ အလုပ်နှစ်ခုအကြား အတိုးအလျှော့ လုပ်တာကို ထိန်းညှိတဲ့ parameter ပါ။ 
*   **High `c` (e.g., 100):** တကယ်လို့ `c` တန်ဖိုးကို "100" ထားမယ်ဆိုရင် SVM မော်ဒယ်က စာကြောင်း တစ်ကြောင်းချင်းစီမှာ ရှိတဲ့ စာလုံးတွေကို tagging လုပ်တဲ့ အပိုင်းကို အလွန်အကျွံအားထည့် မှာ ဖြစ်ပါတယ်။ အဲဒါက လိုင်းချတဲ့အလုပ် ဘော်ဒါသတ်မှတ်တဲ့အလုပ်ကို ပိုမိုရှုပ်ထွေးတာမျိုး ဖြစ်စေပြီး **Overfitting** ဖြစ်စေနိုင်ပါတယ်။ ဆိုလိုတာက training data ကို memorize လုပ်တဲ့အပိုင်းက အားသာသွားပြီး မမြင်ဖူးတဲ့ test data တွေနဲ့ testing လုပ်တဲ့အခါမှာ အမှားတွေလုပ်ပါလိမ့်မယ်။  
*   **Low `c` (e.g., 0.01):** တကယ်လို့ `c` ကို "0.01" လိုမျိုး ထားတဲ့အခါမျိုးမှာတော့ training လုပ်စဉ်မှာ အမှားတချို့ကိုလည်း လက်ခံတတ်တာမျိုး ရှိပြီး လိုင်း (သို့) ဘောဒါ သတ်မှတ်တဲ့အပိုင်းမှာ ပို ပျော့ပျော့ပြောင်းပြောင်း (i.e. generalization) ရှိပါတယ်။ Overfitting ပြဿနာကို မဖြစ်အောင် တားဆီးပေးနိုင်ပါတယ်။ တစ်ခု သတိထားရမှာက ကိုယ့်ဒေတာအခြေအနေပေါ်မူတည်ပြီး ပေးလိုက်တဲ့ `c` တန်ဖိုးက သိပ်နည်းနေရင်တော့ **Underfitting** (ဥပမာ တွေ့သမျှ စာလုံးတိုင်းကို "နမ်" ဆိုပြီး tag လုပ်လိုက်တာမျိုး) ပြဿနာလည်း ရှိပါတယ်။

အထက်က shell script မှာတော့ `-c 1.0` ဆိုပြီး default setting နဲ့ပဲ သွားထားပါတယ်။ လက်တွေ့မှာ အကောင်းဆုံး accuracy ကို အသေးစိတ် ရှာဖွေကြည့်ချင်ရင်တော့ `-c 0.1`, `-c 10`, and `-c 100` ဆိုပြီး ကစားသွားရတဲ့ အခါမျိုးလည်း လုပ်လေ့ရှိကြပါတယ်။

#### **`-e epsilon` : Stopping Tolerance**

ဒီ `-e` option ကတော့ optimizer ကို ဘယ်အချိန်မှာ ရပ်သင့်တယ် ဆိုတာကို control လုပ်တဲ့ အပိုင်းပါ။  

သိကြတဲ့ အတိုင်းပဲ optimization ဆိုတာ ပရိုဂရမ်းမင်း စကားနဲ့ ပြောရရင် lopping ပါပဲ။ ထပ်ခါထပ်ခါ လုပ်ရတဲ့ process ပါ။ အဲဒါကြောင့် ဆရာတို့က threshold တစ်ခု သတ်မှတ်ပြီး အဲဒီ တန်ဖိုးအောက် accuracy or performance က ကျနေပြီ ဆိုရင်တော့ ရပ်လိုက်တော့ဆိုပြီး သတ်မှတ်ပေးလို့ ရပါတယ်။  
*   `-e` တန်ဖိုးကို ဥပမာ (e.g., `0.0001`)  ဆိုပြီး သေးတဲ့ တန်ဖိုးပေးထားရင်တော့ ပိုမှန်မှန်ကန်ကန် ခန့်မှန်းပေးတဲ့ မော်ဒယ်ကို ရနိုင်ပါတယ်။ ဒါပေမဲ့ training time ကတော့ ကြာပါလိမ့်မယ်။  
*   တကယ်လို့ `-e` တန်ဖိုးကို ကြီးကြီး (e.g., `0.1`) ထားပြီး training လုပ်ရင်တော့ training time က မြန်ပါလိမ့်မယ်။ ဒါပေမဲ့ မော်ဒယ်က အကောင်းဆုံး performance ကို မရသေးခင်မှာ training လုပ်တဲ့ process က ရပ်သွားတဲ့ အခြေအနေမျိုးလည်း ဖြစ်ပေါ်စေနိုင်ပါတယ်။  

* ဆရာကတော့ လက်ရှိ running script မှာ `-e 0.001` ထားကြည့်ထားပါတယ်။ အခြေခံအားဖြင့် training time နဲ့ model performance ကို balance မျှအောင် ထားထားတဲ့ သဘောပါ။  

#### **`-B bias` : The Bias Term**
If you use `-B 1`, LIBLINEAR adds an extra feature (index 0) to every single word with a value of 1. 

Modeling လုပ်တဲ့ ထုံးစံအတိုင်း ဒီ option က Bias ဖြည့်တာပါ။  ဥပမာ `-B 1` ဆိုတဲ့ setting လုပ်ထားရင် LIBLINEAR က `index 0` ဆိုပြီး စာလုံးတိုင်း (i.e. word) ကို တန်ဖိုး `1` ပေးထားတဲ့ feature အပိုတစ်ခု (extra feature) ထပ်ဖြည့်တာပါ။  

*   ဘာကြောင့် အဲဒီလို လုပ်တာလဲ ဆိုတော့ decision boundary ကို ပထမ ရှိနေတဲ့ boundary နဲ့ ခပ်လှမ်းလှမ်းမှာ စချတဲ့ သဘောလို့ ဆရာ နားလည်ပါတယ်။  

*   ဆရာတို့ ခုလုပ်နေတဲ့ POS tagging လို ကိစ္စမျိုးမှာ ဆိုရင် (`/punc` သို့မဟုတ် `/n`)  လို tag တွေက (`/conj`) လို tag တွေထက် စာရင် ကောပတ်စ် ထဲမှာ အကြိမ်အရေအတွက် ပိုများလေ့ရှိပါတယ်။ အဲဒါကြောင့် frequency နည်းတဲ့ tag တွေအတွက် မျက်နှာလိုက်တာမျိုး မဖြစ်ရလေအောင် bias term ထည့်ပြီး ညှိပေးတဲ့ သဘောပါ။  

လက်ရှိ shell script မှာတော့ ဆရာ  `-B 1` ကို မသုံးထားပါဘူး။ နောက်ပိုင်းကျမှ automatic tuning လိုမျိုး လုပ်ပြဖို့လည်း စဉ်းစားထားလို့ပါ။  

#### **`-wi weight` : Class Weights**

တကယ်လို့ ဆရာတို့ corpus မှာ ရှိတဲ့ tag တွေက imbalance တအားဖြစ်နေတယ် ဆိုကြပါစို့။ ဥပမာ (50% Noun ရှိပြီး 1% Conjunction ပဲ ရှိတယ်) ဆိုတဲ့ အခြေအနေမျိုး ဆိုရင် မော်ဒယ်က Conjunction tag တွေကို သိပ် ဂရုမစိုက်တော့တာမျိုးလည်း ဖြစ်သွားနိုင်ပါတယ်။  ဒီကိစ္စမျိုးမှာတော့ accuracy ကိုပဲ ကြည့်ပြီး မော်ဒယ်က ကောင်းတယ် မကောင်းဘူး ဆုံးဖြတ်လို့ မရပါဘူး။  

*   ဥပမာ `-w1 1 -w2 5` လို့ ထားကြည့်ရအောင်။ အဲဒါဆိုရင် မော်ဒယ်က Class 2 word ကို tagging မှားလုပ်မိရင်၊ Class 1 word ကို မှားလုပ်မိတာထက် ၅ဆ ပို အပြစ်ပေးပါ (penelize) လို့ ဆိုလိုပါတယ်။ 

* **POS tagging လုပ်တဲ့အခါမှာတော့ ဘယ် tag တွေက နည်းနေတယ် ပြီးတော့ မော်ဒယ်ကလည်း အဲဒီ tag တွေနဲ့ ပတ်သက်ရင် ဂရုမစိုက်ဘူးဆိုတာကို သေသေချာချာ သိမှသာ ဒီ `-wi weight` option ကို သုံးပါလို့ အကြံပြုချင်ပါတယ်။**

### `predict` Command Parameters

Predict command ကတော့ train command ထက် စာရင် option ကနည်းနည်းလေးပါပဲ။  `test_file`, `model_file`, `output_file` ပေးရမယ်။ option အနေနဲ့ကတော့ `-b` နဲ့ `-q` ပဲ ရှိတာပါ။  

-b probability_estimates
Default တန်ဖိုးက `0` ပါ။ အဲဒီလို ထားထားရင်တော့ သူက ခန့်မှန်း သို့မဟုတ် classification လုပ်ပြီးရလာတဲ့ class ID တွေကိုပဲ ရိုက်ထုတ်ပေးမှာပါ။  
တကယ်လို့ ကိုယ်က SVM မော်ဒယ်ကို training လုပ်တဲ့အခါမှာ -s 0 (Logistic Regression) နဲ့ train ထားပြီး၊ prediငt (or) test လုပ်တဲ့အခါမှာ `-b 1` ထားပေးလိုက်ရင်တော့ မော်ဒယ်က ခန့်မှန်းပြီးရလာတဲ့ class ID အပြင် သူ့နောက်မှာ class အားလုံးအတွက် probability score တွေကို ရိုက်ထုတ်ပြပေးမှာ ဖြစ်ပါတယ်။  

ဆရာတို့က `-s 2` ထားပြီး training လုပ်ထားရင်တော့ predict လုပ်တဲ့အခါမှာ `-b 1` ထားထားလည်း probability တွေကို ရိုက်ထုတ်ပြပေးနိုင်မှာ မဟုတ်ပါဘူး။ 

Note: **ဘာကြောင့်လဲ ဆိုတော့ SVM ရဲ့ အလုပ်လုပ်ပုံက probability တွက်တာ မဟုတ်လို့**

Question: အဲဒါဆိုရင် SVM က တကယ်တမ်းက ဘယ်လို အလုပ်လုပ်တာလဲ။ စဉ်းစားကြည့်ပြီး ဆရာ့ကို ရှင်းပြပေးပါလား။  
Hint: raw distance score (+5.2 or -3.8), binary classifier, One-vs-Rest (OvR)


## Manual Experiments
### Experiment 1: Increasing the Cost (-c)

Let's see if making the model try harder to classify training data improves or hurts test accuracy.  

In [84]:
# 1. Train with higher cost C=10
!time ./liblinear/train -s 2 -c 10.0 -e 0.001 ./work/train.svm ./work/model_c10.bin

init f 5.645e+06 |g| 1.331e+07
iter  1 f 2.006e+04 |g| 3.767e+05 CG   2 step_size 1.00e+00 
iter  2 f 4.445e+03 |g| 1.045e+05 CG   2 step_size 1.00e+00 
iter  3 f 1.183e+03 |g| 2.281e+04 CG   4 step_size 1.00e+00 
iter  4 f 6.539e+02 |g| 5.666e+03 CG   6 step_size 1.00e+00 
iter  5 f 5.690e+02 |g| 6.452e+03 CG  19 step_size 2.50e-01 
iter  6 f 4.485e+02 |g| 5.349e+03 CG  18 step_size 5.00e-01 
iter  7 f 3.977e+02 |g| 1.449e+03 CG   4 step_size 1.00e+00 
iter  8 f 3.757e+02 |g| 2.114e+03 CG  21 step_size 1.25e-01 
iter  9 f 3.297e+02 |g| 2.681e+03 CG  19 step_size 5.00e-01 
iter 10 f 2.870e+02 |g| 1.046e+03 CG   3 step_size 1.00e+00 
iter 11 f 2.784e+02 |g| 1.616e+03 CG  15 step_size 2.50e-01 
iter 12 f 2.502e+02 |g| 4.833e+02 CG   3 step_size 1.00e+00 
iter 13 f 2.436e+02 |g| 9.728e+02 CG  14 step_size 2.50e-01 
iter 14 f 2.208e+02 |g| 2.910e+02 CG   3 step_size 1.00e+00 
iter 15 f 2.176e+02 |g| 7.915e+02 CG  13 step_size 2.50e-01 
iter 16 f 1.966e+02 |g| 1.957e+02 CG   3 step_size 1.0

In [85]:
# 2. Predict
!time ./liblinear/predict ./work/test.svm ./work/model_c10.bin ./work/test.pred_c10

Accuracy = 99.4802% (13398/13468)

real	0m0.639s
user	0m0.614s
sys	0m0.025s


In [86]:
# 3. Evaluate
!python3 ./04_evaluate.py ./work/test.conll ./work/test.pred_c10 ./work/vocab.pkl


=== Overall Accuracy ===
  13398/13468 = 99.48%

Tag         Precision     Recall         F1     TP     FP     FN
----------------------------------------------------------------
abb           100.00%    100.00%    100.00%     12      0      0
adj            96.77%     98.09%     97.42%    359     12      7
adv            99.24%    100.00%     99.62%    262      2      0
conj          100.00%     99.76%     99.88%    410      0      1
fw            100.00%    100.00%    100.00%     87      0      0
int           100.00%    100.00%    100.00%     25      0      0
n              99.63%     99.47%     99.55%   2984     11     16
num           100.00%    100.00%    100.00%    155      0      0
part           99.37%     99.47%     99.42%   3172     20     17
ppm            99.85%     99.71%     99.78%   2054      3      6
pron           99.37%     98.95%     99.16%    471      3      5
punc          100.00%    100.00%    100.00%   1270      0      0
sb            100.00%    100.00%    100.

### Experiment 2: Lowering the Cost (-c)

Let's see if more regularization (allowing training errors) helps generalization.  

In [88]:
# 1. Train with lower cost C=0.1
!time ./liblinear/train -s 2 -c 0.1 -e 0.001 ./work/train.svm ./work/model_c01.bin

init f 5.645e+04 |g| 1.331e+05
iter  1 f 5.417e+02 |g| 5.784e+03 CG   2 step_size 1.00e+00 
iter  2 f 1.228e+02 |g| 1.729e+03 CG   3 step_size 1.00e+00 
iter  3 f 6.801e+01 |g| 6.254e+02 CG   2 step_size 1.00e+00 
iter  4 f 3.116e+01 |g| 2.084e+02 CG   8 step_size 1.00e+00 
iter  5 f 2.808e+01 |g| 7.976e+01 CG   2 step_size 1.00e+00 
iter  6 f 2.142e+01 |g| 2.831e+02 CG  15 step_size 1.00e+00 
iter  7 f 1.864e+01 |g| 7.418e+01 CG   2 step_size 1.00e+00 
iter  8 f 1.680e+01 |g| 2.719e+01 CG  11 step_size 1.00e+00 
iter  9 f 1.591e+01 |g| 1.637e+01 CG  17 step_size 1.00e+00 
iter 10 f 1.541e+01 |g| 8.898e+00 CG   6 step_size 1.00e+00 
iter 11 f 1.480e+01 |g| 4.406e+00 CG  16 step_size 1.00e+00 
iter 12 f 1.466e+01 |g| 2.056e+00 CG  13 step_size 1.00e+00 
iter 13 f 1.461e+01 |g| 7.161e-01 CG  13 step_size 1.00e+00 
init f 5.645e+04 |g| 8.034e+04
iter  1 f 1.143e+04 |g| 7.250e+03 CG   2 step_size 1.00e+00 
iter  2 f 5.151e+03 |g| 2.257e+03 CG   5 step_size 1.00e+00 
iter  3 f 4.521e+03 |g|

In [89]:
# 2. Predict
!time ./liblinear/predict ./work/test.svm ./work/model_c01.bin ./work/test.pred_c01

Accuracy = 98.1734% (13222/13468)

real	0m0.667s
user	0m0.641s
sys	0m0.026s


In [90]:
# 3. Evaluate
!python3 ./04_evaluate.py ./work/test.conll ./work/test.pred_c01 ./work/vocab.pkl


=== Overall Accuracy ===
  13222/13468 = 98.17%

Tag         Precision     Recall         F1     TP     FP     FN
----------------------------------------------------------------
abb           100.00%    100.00%    100.00%     12      0      0
adj            92.92%     89.62%     91.24%    328     25     38
adv            94.57%     93.13%     93.85%    244     14     18
conj           94.80%     97.57%     96.16%    401     22     10
fw            100.00%    100.00%    100.00%     87      0      0
int           100.00%    100.00%    100.00%     25      0      0
n              98.76%     98.60%     98.68%   2958     37     42
num           100.00%     98.71%     99.35%    153      0      2
part           97.97%     98.37%     98.17%   3137     65     52
ppm            98.79%     99.17%     98.98%   2043     25     17
pron           97.88%     97.06%     97.47%    462     10     14
punc          100.00%    100.00%    100.00%   1270      0      0
sb            100.00%    100.00%    100.

### Experiment 3: Adding a Bias Term (-B)

Sometimes shifting the decision boundary helps.  

In [91]:
# 1. Train with bias term enabled
!time ./liblinear/train -s 2 -c 1.0 -e 0.001 -B 1 ./work/train.svm ./work/model_bias.bin

init f 5.645e+05 |g| 1.730e+06
iter  1 f 2.448e+03 |g| 5.245e+04 CG   2 step_size 1.00e+00 
iter  2 f 6.862e+02 |g| 1.623e+04 CG   2 step_size 1.00e+00 
iter  3 f 1.967e+02 |g| 3.892e+03 CG   5 step_size 1.00e+00 
iter  4 f 1.532e+02 |g| 1.367e+03 CG   2 step_size 1.00e+00 
iter  5 f 9.984e+01 |g| 6.554e+02 CG  12 step_size 1.00e+00 
iter  6 f 8.720e+01 |g| 2.277e+02 CG   4 step_size 1.00e+00 
iter  7 f 7.758e+01 |g| 3.565e+02 CG  16 step_size 5.00e-01 
iter  8 f 6.665e+01 |g| 1.457e+02 CG   6 step_size 1.00e+00 
iter  9 f 6.438e+01 |g| 2.264e+02 CG  17 step_size 1.00e+00 
iter 10 f 5.244e+01 |g| 7.465e+01 CG   6 step_size 1.00e+00 
iter 11 f 5.078e+01 |g| 7.833e+01 CG  17 step_size 5.00e-01 
iter 12 f 4.773e+01 |g| 2.597e+01 CG   6 step_size 1.00e+00 
iter 13 f 4.682e+01 |g| 4.800e+01 CG  14 step_size 1.00e+00 
iter 14 f 4.414e+01 |g| 1.593e+01 CG   4 step_size 1.00e+00 
init f 5.645e+05 |g| 1.026e+06
iter  1 f 8.220e+04 |g| 8.304e+04 CG   2 step_size 1.00e+00 
iter  2 f 3.248e+04 |g|

In [92]:
# 2. Predict
! time ./liblinear/predict ./work/test.svm ./work/model_bias.bin ./work/test.pred_bias

Accuracy = 99.302% (13374/13468)

real	0m0.682s
user	0m0.654s
sys	0m0.028s


In [93]:
# 3. Evaluate
!python3 ./04_evaluate.py ./work/test.conll ./work/test.pred_bias ./work/vocab.pkl


=== Overall Accuracy ===
  13374/13468 = 99.30%

Tag         Precision     Recall         F1     TP     FP     FN
----------------------------------------------------------------
abb           100.00%    100.00%    100.00%     12      0      0
adj            96.49%     97.54%     97.01%    357     13      9
adv            99.24%     99.24%     99.24%    260      2      2
conj           99.27%     99.76%     99.51%    410      3      1
fw            100.00%    100.00%    100.00%     87      0      0
int           100.00%    100.00%    100.00%     25      0      0
n              99.50%     99.37%     99.43%   2981     15     19
num           100.00%    100.00%    100.00%    155      0      0
part           99.06%     99.44%     99.25%   3171     30     18
ppm            99.76%     99.51%     99.64%   2050      5     10
pron           99.16%     98.74%     98.95%    470      4      6
punc          100.00%    100.00%    100.00%   1270      0      0
sb            100.00%    100.00%    100.

### Experiment 4: Trying Logistic Regression (-s 0)

Logistic Regression uses a different loss function (log loss instead of hinge loss).  

In [94]:
# 1. Train with Logistic Regression
!time ./liblinear/train -s 0 -c 1.0 -e 0.001 ./work/train.svm ./work/model_logreg.bin

init f 3.913e+05 |g| 3.328e+05
iter  1 f 7.502e+04 |g| 8.149e+04 CG   2 step_size 1.00e+00 
iter  2 f 2.611e+04 |g| 2.907e+04 CG   2 step_size 1.00e+00 
iter  3 f 9.954e+03 |g| 1.079e+04 CG   2 step_size 1.00e+00 
iter  4 f 4.119e+03 |g| 4.072e+03 CG   2 step_size 1.00e+00 
iter  5 f 1.931e+03 |g| 1.544e+03 CG   2 step_size 1.00e+00 
iter  6 f 1.086e+03 |g| 5.772e+02 CG   2 step_size 1.00e+00 
iter  7 f 7.536e+02 |g| 2.081e+02 CG   2 step_size 1.00e+00 
iter  8 f 5.214e+02 |g| 1.120e+02 CG   4 step_size 1.00e+00 
iter  9 f 3.562e+02 |g| 6.680e+01 CG   6 step_size 1.00e+00 
iter 10 f 3.317e+02 |g| 1.946e+01 CG   2 step_size 1.00e+00 
iter 11 f 3.169e+02 |g| 5.593e+00 CG   6 step_size 1.00e+00 
iter 12 f 3.158e+02 |g| 1.550e+00 CG   4 step_size 1.00e+00 
init f 3.913e+05 |g| 2.009e+05
iter  1 f 1.359e+05 |g| 5.104e+04 CG   2 step_size 1.00e+00 
iter  2 f 7.295e+04 |g| 1.752e+04 CG   3 step_size 1.00e+00 
iter  3 f 4.757e+04 |g| 7.297e+03 CG   4 step_size 1.00e+00 
iter  4 f 3.536e+04 |g|

In [95]:
# 2. Predict
!time ./liblinear/predict ./work/test.svm ./work/model_logreg.bin ./work/test.pred_logreg

Accuracy = 97.9804% (13196/13468)

real	0m0.648s
user	0m0.628s
sys	0m0.019s


In [96]:
# 3. Evaluate
!python3 ./04_evaluate.py ./work/test.conll ./work/test.pred_logreg ./work/vocab.pkl


=== Overall Accuracy ===
  13196/13468 = 97.98%

Tag         Precision     Recall         F1     TP     FP     FN
----------------------------------------------------------------
abb           100.00%    100.00%    100.00%     12      0      0
adj            93.33%     87.98%     90.58%    322     23     44
adv            94.94%     93.13%     94.03%    244     13     18
conj           94.35%     97.57%     95.93%    401     24     10
fw            100.00%    100.00%    100.00%     87      0      0
int           100.00%     96.00%     97.96%     24      0      1
n              98.30%     98.53%     98.42%   2956     51     44
num           100.00%     98.71%     99.35%    153      0      2
part           97.75%     98.28%     98.01%   3134     72     55
ppm            98.65%     99.08%     98.86%   2041     28     19
pron           97.88%     97.06%     97.47%    462     10     14
punc          100.00%    100.00%    100.00%   1270      0      0
sb            100.00%    100.00%    100.

### Experiment 5: The "Right" Way to Tune (Cross-Validation)

We shouldn't tune C by looking at the test set. We should use Cross-Validation on the training set. This command doesn't output a model file; it just splits the training set into 5 parts, trains on 4, and tests on 1, repeating 5 times.  

In [97]:
# Run 5-fold cross validation on training data
!time ./liblinear/train -s 2 -c 1.0 -e 0.001 -v 5 ./work/train.svm

init f 4.516e+05 |g| 1.043e+06
iter  1 f 5.365e+03 |g| 3.618e+04 CG   2 step_size 1.00e+00 
iter  2 f 2.011e+03 |g| 1.099e+04 CG   4 step_size 1.00e+00 
iter  3 f 8.330e+02 |g| 4.300e+03 CG   8 step_size 1.00e+00 
iter  4 f 5.235e+02 |g| 1.638e+03 CG  11 step_size 1.00e+00 
iter  5 f 4.181e+02 |g| 8.894e+02 CG  17 step_size 1.00e+00 
iter  6 f 3.503e+02 |g| 6.092e+02 CG  20 step_size 1.00e+00 
iter  7 f 2.991e+02 |g| 3.702e+02 CG  21 step_size 1.00e+00 
iter  8 f 2.676e+02 |g| 2.728e+02 CG  22 step_size 1.00e+00 
iter  9 f 2.477e+02 |g| 8.244e+01 CG  21 step_size 1.00e+00 
iter 10 f 2.414e+02 |g| 1.090e+02 CG  21 step_size 1.00e+00 
iter 11 f 2.353e+02 |g| 3.486e+01 CG   6 step_size 1.00e+00 
iter 12 f 2.351e+02 |g| 1.472e+02 CG  20 step_size 1.00e+00 
iter 13 f 2.318e+02 |g| 7.137e+01 CG   2 step_size 1.00e+00 
iter 14 f 2.295e+02 |g| 2.307e+01 CG   4 step_size 1.00e+00 
iter 15 f 2.287e+02 |g| 3.279e+01 CG  22 step_size 2.50e-01 
iter 16 f 2.270e+02 |g| 2.451e+01 CG  22 step_size 1.0

### Let's Run 5-fold Corss Validation with c=10.0

In [98]:
# Run 5-fold cross validation on training data
!./liblinear/train -s 2 -c 10.0 -e 0.001 -v 5 ./work/train.svm  

init f 4.516e+06 |g| 1.043e+07
iter  1 f 4.002e+04 |g| 2.893e+05 CG   2 step_size 1.00e+00 
iter  2 f 1.038e+04 |g| 7.970e+04 CG   4 step_size 1.00e+00 
iter  3 f 3.738e+03 |g| 2.697e+04 CG   5 step_size 1.00e+00 
iter  4 f 2.146e+03 |g| 9.137e+03 CG   6 step_size 1.00e+00 
iter  5 f 1.647e+03 |g| 7.378e+03 CG   9 step_size 1.00e+00 
iter  6 f 1.401e+03 |g| 2.442e+03 CG   4 step_size 1.00e+00 
iter  7 f 1.289e+03 |g| 4.840e+03 CG  20 step_size 2.50e-01 
iter  8 f 1.125e+03 |g| 1.522e+03 CG   4 step_size 1.00e+00 
iter  9 f 1.068e+03 |g| 3.795e+03 CG  21 step_size 2.50e-01 
iter 10 f 9.442e+02 |g| 1.122e+03 CG   3 step_size 1.00e+00 
iter 11 f 8.895e+02 |g| 2.374e+03 CG  20 step_size 2.50e-01 
iter 12 f 7.947e+02 |g| 5.294e+02 CG   5 step_size 1.00e+00 
iter 13 f 7.836e+02 |g| 1.942e+03 CG  17 step_size 2.50e-01 
iter 14 f 7.037e+02 |g| 4.573e+02 CG   4 step_size 1.00e+00 
iter 15 f 7.017e+02 |g| 1.461e+03 CG  15 step_size 2.50e-01 
iter 16 f 6.416e+02 |g| 3.477e+02 CG   4 step_size 1.0

### Let's Run 5-fold Cross Validation with c=0.1

In [100]:
# Run 5-fold cross validation on training data
!time ./liblinear/train -s 2 -c 0.1 -e 0.001 -v 5 ./work/train.svm

init f 4.516e+04 |g| 1.043e+05
iter  1 f 7.840e+02 |g| 4.657e+03 CG   2 step_size 1.00e+00 
iter  2 f 3.306e+02 |g| 1.344e+03 CG   4 step_size 1.00e+00 
iter  3 f 1.995e+02 |g| 5.615e+02 CG   7 step_size 1.00e+00 
iter  4 f 1.418e+02 |g| 2.959e+02 CG  11 step_size 1.00e+00 
iter  5 f 1.341e+02 |g| 1.104e+02 CG   2 step_size 1.00e+00 
iter  6 f 1.103e+02 |g| 1.008e+02 CG  13 step_size 1.00e+00 
iter  7 f 1.004e+02 |g| 4.226e+01 CG  17 step_size 1.00e+00 
iter  8 f 9.660e+01 |g| 2.167e+01 CG  15 step_size 1.00e+00 
iter  9 f 9.476e+01 |g| 7.941e+00 CG  17 step_size 1.00e+00 
iter 10 f 9.406e+01 |g| 7.035e+00 CG  17 step_size 1.00e+00 
iter 11 f 9.385e+01 |g| 2.035e+00 CG  16 step_size 1.00e+00 
iter 12 f 9.378e+01 |g| 1.715e+00 CG  16 step_size 1.00e+00 
iter 13 f 9.377e+01 |g| 9.618e-01 CG   3 step_size 1.00e+00 
init f 4.516e+04 |g| 6.418e+04
iter  1 f 9.479e+03 |g| 5.892e+03 CG   2 step_size 1.00e+00 
iter  2 f 4.272e+03 |g| 1.813e+03 CG   5 step_size 1.00e+00 
iter  3 f 2.771e+03 |g|

## Train with c=0.1 and Save 

In [101]:
!time ./liblinear/train -s 2 -c 1.0 -e 0.001 ./work/train.svm ./work/model_c1.0.noBias.bin

init f 5.645e+05 |g| 1.331e+06
iter  1 f 2.816e+03 |g| 4.453e+04 CG   2 step_size 1.00e+00 
iter  2 f 7.823e+02 |g| 1.384e+04 CG   2 step_size 1.00e+00 
iter  3 f 3.619e+02 |g| 4.850e+03 CG   2 step_size 1.00e+00 
iter  4 f 1.600e+02 |g| 1.573e+03 CG   7 step_size 1.00e+00 
iter  5 f 1.238e+02 |g| 5.038e+02 CG   3 step_size 1.00e+00 
iter  6 f 9.560e+01 |g| 5.852e+02 CG  14 step_size 1.00e+00 
iter  7 f 7.563e+01 |g| 1.791e+02 CG   5 step_size 1.00e+00 
iter  8 f 6.846e+01 |g| 2.279e+02 CG  17 step_size 5.00e-01 
iter  9 f 6.228e+01 |g| 7.609e+01 CG   4 step_size 1.00e+00 
iter 10 f 5.737e+01 |g| 1.046e+02 CG  16 step_size 5.00e-01 
iter 11 f 5.319e+01 |g| 4.253e+01 CG   6 step_size 1.00e+00 
iter 12 f 5.280e+01 |g| 7.147e+01 CG  16 step_size 1.00e+00 
iter 13 f 4.669e+01 |g| 2.775e+01 CG   4 step_size 1.00e+00 
iter 14 f 4.618e+01 |g| 2.915e+01 CG  15 step_size 2.50e-01 
iter 15 f 4.495e+01 |g| 8.018e+00 CG   6 step_size 1.00e+00 
init f 5.645e+05 |g| 8.034e+05
iter  1 f 8.063e+04 |g|

In [102]:
# 2. Predict
!time ./liblinear/predict ./work/test.svm ./work/model_c1.0.noBias.bin ./work/test.pred_logc1.0.noBias

Accuracy = 99.3169% (13376/13468)

real	0m0.666s
user	0m0.631s
sys	0m0.035s


In [104]:
# 3. Evaluate
!python3 ./04_evaluate.py ./work/test.conll ./work/test.pred_logc1.0.noBias ./work/vocab.pkl


=== Overall Accuracy ===
  13376/13468 = 99.32%

Tag         Precision     Recall         F1     TP     FP     FN
----------------------------------------------------------------
abb           100.00%    100.00%    100.00%     12      0      0
adj            96.49%     97.54%     97.01%    357     13      9
adv            99.24%     99.24%     99.24%    260      2      2
conj           99.27%     99.76%     99.51%    410      3      1
fw            100.00%    100.00%    100.00%     87      0      0
int           100.00%    100.00%    100.00%     25      0      0
n              99.50%     99.37%     99.43%   2981     15     19
num           100.00%    100.00%    100.00%    155      0      0
part           99.12%     99.44%     99.28%   3171     28     18
ppm            99.76%     99.51%     99.64%   2050      5     10
pron           99.16%     98.74%     98.95%    470      4      6
punc          100.00%    100.00%    100.00%   1270      0      0
sb            100.00%    100.00%    100.

## Check Model Filesize

In [105]:
!ls -lh ./work/*.bin

-rw-rw-r-- 1 ye ye 111M Jul 28 16:32 ./work/model_bias.bin
-rw-rw-r-- 1 ye ye 111M Jul 28 14:37 ./work/model.bin
-rw-rw-r-- 1 ye ye 111M Jul 28 16:30 ./work/model_c01.bin
-rw-rw-r-- 1 ye ye 108M Jul 28 16:28 ./work/model_c10.bin
-rw-rw-r-- 1 ye ye 111M Jul 28 16:53 ./work/model_c1.0.noBias.bin
-rw-rw-r-- 1 ye ye 111M Jul 28 16:36 ./work/model_logreg.bin


## Summary

အချိန်ပေးပြီး လက်တွေ့ run ပြထားပါတယ်။  
ဒီ tutorial ကနေ LIBLINEAR ကိုသုံးပြီး SVM based POS tagger ဆောက်တာရယ်၊ အကောင်းဆုံး မော်ဒယ်ကို ရရှိဖို့အတွက် ဘယ်လို training parameter တွေနဲ့ tuning လုပ်လို့ရနိုင်တယ် ဆိုတာကို ကျောင်းသား/သူတွေအနေနဲ့ အများကြီး ဗဟုသုတ ရရှိသွားမယ်လို့ ယုံကြည်ပါတယ်။  